In [9]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.utils import resample
import tune
from torch.profiler import profile, record_function, ProfilerActivity

In [10]:
pipe, x, y = tune.process_data()

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(x, y, 
                    test_size=0.20,
                    stratify=y,
                    random_state=1)

X_train = pipe.fit_transform(X_train, y_train)
X_test = pipe.transform(X_test)
y_train = y_train.to_numpy()
y_train = y_train.reshape(-1, 1)
y_test = y_test.to_numpy()
y_test = y_test.reshape(-1, 1)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [13]:
from torch.utils.data import DataLoader, TensorDataset

X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()
X_train = X_train.to(device)
y_train = y_train.to(device)
train_ds = TensorDataset(X_train, y_train)

batch_size = 256
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()
X_test = X_test.to(device)
y_test = y_test.to(device)
test_dl = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

In [14]:
input_size = X_train.shape[1]
hidden_units = [input_size, input_size]

all_layers = nn.ModuleList()
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit
all_layers.append(nn.Linear(hidden_units[-1], 1)) 
all_layers.append(nn.Sigmoid())  
model = nn.Sequential(*all_layers)
model.to(device)

Sequential(
  (0): Linear(in_features=14, out_features=14, bias=True)
  (1): ReLU()
  (2): Linear(in_features=14, out_features=14, bias=True)
  (3): ReLU()
  (4): Linear(in_features=14, out_features=1, bias=True)
  (5): Sigmoid()
)

In [15]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            pred = model(x_batch.to(device))
            is_correct = ((pred > 0.5).float() == y_batch.to(device)).float()
            correct += is_correct.sum().cpu()
            total += y_batch.size(0)
    accuracy = correct / total
    return accuracy

In [22]:
#class_weights = torch.tensor([0.9]).to(device)
loss_fn = nn.BCELoss()  
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

torch.manual_seed(1)
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    accuracy_hist_train = 0
    training_history = []
    validation_history = []
    for x_batch, y_batch in train_dl:
        pred = model(x_batch.to(device))
        loss = loss_fn(pred, y_batch.to(device))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = ((pred > 0.5).float() == y_batch.to(device)).float()
        accuracy_hist_train += is_correct.sum().cpu()
    accuracy_hist_train /= len(train_dl.dataset)
    validation_accuracy = evaluate(model, test_dl, device)
    training_history.append(accuracy_hist_train)
    validation_history.append(validation_accuracy)
    print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}  Validation Accuracy {validation_accuracy:.4f}')

Epoch 0  Accuracy 0.8837  Validation Accuracy 0.8816
Epoch 1  Accuracy 0.8835  Validation Accuracy 0.8824
Epoch 2  Accuracy 0.8838  Validation Accuracy 0.8820
Epoch 3  Accuracy 0.8837  Validation Accuracy 0.8824
Epoch 4  Accuracy 0.8838  Validation Accuracy 0.8821
Epoch 5  Accuracy 0.8836  Validation Accuracy 0.8824
Epoch 6  Accuracy 0.8838  Validation Accuracy 0.8822
Epoch 7  Accuracy 0.8835  Validation Accuracy 0.8826
Epoch 8  Accuracy 0.8837  Validation Accuracy 0.8825
Epoch 9  Accuracy 0.8838  Validation Accuracy 0.8825
Epoch 10  Accuracy 0.8838  Validation Accuracy 0.8825
Epoch 11  Accuracy 0.8837  Validation Accuracy 0.8828
Epoch 12  Accuracy 0.8838  Validation Accuracy 0.8829
Epoch 13  Accuracy 0.8838  Validation Accuracy 0.8825
Epoch 14  Accuracy 0.8840  Validation Accuracy 0.8828
Epoch 15  Accuracy 0.8836  Validation Accuracy 0.8823
Epoch 16  Accuracy 0.8838  Validation Accuracy 0.8822
Epoch 17  Accuracy 0.8837  Validation Accuracy 0.8824
Epoch 18  Accuracy 0.8838  Validation 

In [24]:
model.eval()
model.to('cpu')
# X_test_t = torch.tensor(X_test).float().to('cpu')
X_test_t = X_test.to('cpu')
y_test = y_test.to('cpu')
with torch.no_grad():
    y_pred = model(X_test_t)
y_pred = (y_pred > 0.5).float().numpy()
y_test = y_test.numpy().astype(int)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.93      0.93     70352
           1       0.71      0.69      0.70     17476

    accuracy                           0.88     87828
   macro avg       0.82      0.81      0.81     87828
weighted avg       0.88      0.88      0.88     87828



In [25]:
from sklearn.metrics import roc_auc_score
auc_score = roc_auc_score(y_test, y_pred)
print(f'ROC AUC Score: {auc_score:.4f}')

ROC AUC Score: 0.8097


In [ ]:
testing_data = pd.read_csv("./data/test.csv")
preds = model(torch.tensor(pipe.transform(testing_data.drop(columns=["id"]))).float().to(device)).cpu().detach().numpy()
submission = pd.DataFrame({"id": testing_data["id"], "target": preds.flatten()})
submission.to_csv("./data/nn_baseline_submission.csv", index=False)